In [1]:
import torch
import numpy as np
from adapted_sgformer.scripts.train_detection import load_config
from adaptedsgformer.models.detection_models import DetectionGT
from adaptedsgformer.utils import format_data, embed_1D_scalar
from pathlib import Path
from dagr.data.ncaltech101_data import NCaltech101
from torch_geometric.loader import DataLoader
from dagr.data.augment import Augmentations
from argparse import Namespace
from dagr.model.utils import postprocess_network_output, convert_to_training_format
from yolox.models import YOLOX
from torch_scatter import scatter

In [2]:
cfg_path = 'config/detection/config_dagt_gnn.yaml'

In [3]:
cfg = load_config(cfg_path)

In [4]:
dataset_path = Path(cfg["data_directory"]) / cfg["dataset"]
augmentations = Augmentations(Namespace(**cfg["augmentations"]))
dataset = NCaltech101(dataset_path, "training", augmentations.transform_training, num_events=cfg["n_nodes"])

In [5]:
len(dataset)

6559

In [6]:
loader = DataLoader(dataset, follow_batch=['bbox', 'bbox0'], shuffle=True, drop_last=True, **cfg["dataloader"])

In [7]:
model = DetectionGT(num_classes=dataset.num_classes, args=cfg["model_params"], height=dataset.height, width=dataset.width).cuda()

poolings: tensor([[0.0179, 0.0250, 1.0000],
        [0.0357, 0.0500, 1.0000],
        [0.0714, 0.1000, 1.0000],
        [0.1429, 0.2000, 1.0000]])
samplings: tensor([2240,  560,  140,   35])


In [8]:
for data in loader:
    data.cuda()
    break

In [16]:
data = format_data(data)
with torch.no_grad():
    outputs = model.forward(data)

In [9]:
data = format_data(data)
targets = convert_to_training_format(data.bbox, data.bbox_batch, data.num_graphs)

In [10]:
with torch.no_grad():
    fpn_outs = model.backbone(data)

In [11]:
model.backbone

BackboneGT(
  (x_embedding): Embedding(2, 24)
  (events_to_graph): EV_TGN()
  (pe_embedding): Sequential(
    (0): Linear(in_features=12, out_features=12, bias=True)
    (1): LeakyReLU(negative_slope=0.01)
  )
  (proj): Linear(in_features=36, out_features=36, bias=True)
  (block_dagt): ModuleList(
    (0): BlockDectectGT(
      (pooling): UniformSampling()
      (proj): Linear(in_features=48, out_features=36, bias=True)
      (pe_embedding): Sequential(
        (0): Linear(in_features=12, out_features=12, bias=True)
        (1): LeakyReLU(negative_slope=0.01)
      )
      (blockGT): BlockGT(
        (proj): Linear(in_features=36, out_features=48, bias=True)
        (norm1): LayerNorm(36, affine=True, mode=graph)
        (trans): TransLayerMultiHead(
          (Wk): Linear(in_features=36, out_features=48, bias=True)
          (Wq): Linear(in_features=36, out_features=48, bias=True)
          (Wv): Linear(in_features=36, out_features=48, bias=True)
          (Wo): Linear(in_features=48,

In [12]:
fpn_outs

DataBatch(x=[280, 64], pos=[280, 3], width=[8], height=[8], time_window=[8], dist_mat=[280, 35], batch=[280], ptr=[9], edge_index=[2, 3240], edge_attr=[3240, 3])

In [13]:
self = model.head

In [14]:
xin = fpn_outs.clone()

In [15]:
        ev_out = dict(outputs=[], origin_preds=[], x_shifts=[], y_shifts=[], expanded_strides=[])

        batch_size = xin.num_graphs
        normalizer = torch.stack([xin.width[0], xin.height[0]], dim=0)

        #Process data for the first stage of detection 
        cls_output, reg_output, obj_output = self.process_feature(xin)

In [16]:
cls_output

Data(x=[280, 100], edge_index=[2, 3240], edge_attr=[3240, 3], pos=[280, 3], batch=[280])

In [17]:
self

SparseYoloxHead(
  (pe_embedding): Sequential(
    (0): Linear(in_features=192, out_features=64, bias=True)
    (1): SiLU()
  )
  (stem): SimpleMLP(
    (block): Sequential(
      (0): Linear(in_features=64, out_features=64, bias=True)
      (1): SiLU()
      (2): Linear(in_features=64, out_features=64, bias=True)
    )
  )
  (cls_conv): SimpleMLP(
    (block): Sequential(
      (0): Linear(in_features=64, out_features=64, bias=True)
      (1): SiLU()
      (2): Linear(in_features=64, out_features=64, bias=True)
    )
  )
  (cls_pred): SimpleMLP(
    (block): Sequential(
      (0): Linear(in_features=64, out_features=100, bias=True)
      (1): SiLU()
      (2): Linear(in_features=100, out_features=100, bias=True)
    )
  )
  (reg_conv): SimpleMLP(
    (block): Sequential(
      (0): Linear(in_features=64, out_features=64, bias=True)
      (1): SiLU()
      (2): Linear(in_features=64, out_features=64, bias=True)
    )
  )
  (reg_pred): SimpleMLP(
    (block): Sequential(
      (0): Line

In [36]:
from adaptedsgformer.layers.heads import GNNHead, SparseYoloxHead

from argparse import Namespace

In [39]:
        head_args = dict(
            num_classes=dataset.num_classes,
            strides=model.backbone.strides,
            in_channels=model.backbone.hidden_channels_list[-model.backbone.num_scales:], 
            args=Namespace(**cfg["model_params"]['head'])
        )
        head = GNNHead(**head_args).cuda()

In [41]:
model.backbone.poolings[-1]

tensor([0.1429, 0.2000, 1.0000])

In [44]:
xin = fpn_outs.clone()
xin.pooling = model.backbone.poolings[-1].cuda()
batch_size = xin.num_graphs
cls_output_orig, reg_output_orig, obj_output_orig = head.process_feature(xin, head.stem1, head.cls_conv1, head.reg_conv1,
                                                        head.cls_pred1, head.reg_pred1, head.obj_pred1, batch_size=batch_size, cache=head.cache)

In [46]:
output_orig = torch.cat([reg_output_orig, obj_output_orig, cls_output_orig], 1)

In [47]:
output_orig.shape

torch.Size([8, 105, 5, 7])

In [45]:
cls_output_orig.shape, reg_output_orig.shape, obj_output_orig.shape

(torch.Size([8, 100, 5, 7]),
 torch.Size([8, 4, 5, 7]),
 torch.Size([8, 1, 5, 7]))

In [ ]:
TODO (training):

- anchors (locations) are no longer pixel locations but node positions:
    -> modify get_output_and_grid & get_geometry_constraint (what stride to choose?)
    -> x_shifts and y_shift become node x and node y

- implem -> how to code unif sampling (esp. accumulation of dropped nodes features)

- solve problems w/ current implem -> pe max period & dim, graph & edge creation, edge attributes

(inference)

In [58]:
outputs = torch.cat(ev_out['outputs'], 1)

In [59]:
outputs.shape

torch.Size([8, 35, 105])

In [60]:
labels = targets

In [61]:
x_shifts, y_shifts, expanded_strides, origin_preds = ev_out['x_shifts'], ev_out['y_shifts'], ev_out['expanded_strides'], \
    ev_out['origin_preds']

In [62]:
labels.shape

torch.Size([8, 100, 5])

In [63]:
        bbox_preds = outputs[:, :, :4]  # [batch, n_anchors_all, 4]
        obj_preds = outputs[:, :, 4:5]  # [batch, n_anchors_all, 1]
        cls_preds = outputs[:, :, 5:]  # [batch, n_anchors_all, n_cls]

        # calculate targets
        nlabel = (labels.sum(dim=2) > 0).sum(dim=1)  # number of objects

        total_num_anchors = outputs.shape[1]
        x_shifts = torch.cat(x_shifts, 1)  # [1, n_anchors_all]
        y_shifts = torch.cat(y_shifts, 1)  # [1, n_anchors_all]
        expanded_strides = torch.cat(expanded_strides, 1)

In [64]:
        cls_targets = []
        reg_targets = []
        l1_targets = []
        obj_targets = []
        fg_masks = []

        num_fg = 0.0
        num_gts = 0.0

        for batch_idx in range(outputs.shape[0]):
            num_gt = int(nlabel[batch_idx])
            num_gts += num_gt
            break

In [65]:
                gt_bboxes_per_image = labels[batch_idx, :num_gt, 1:5]
                gt_classes = labels[batch_idx, :num_gt, 0]
                bboxes_preds_per_image = bbox_preds[batch_idx]

In [66]:
gt_bboxes_per_image

tensor([[ 70., 110., 140., 130.]], device='cuda:0')

In [ ]:
(gt_matched_classes,
    fg_mask,
    pred_ious_this_matching,
    matched_gt_inds,
    num_fg_img,
) = self.get_assignments( 

In [67]:
        fg_mask, geometry_relation = self.get_geometry_constraint(
            batch_idx,
            gt_bboxes_per_image,
            expanded_strides,
            x_shifts,
            y_shifts,
        )

In [68]:
fg_mask

tensor([ True,  True,  True,  True,  True,  True,  True,  True, False,  True,
        False,  True,  True,  True,  True, False,  True,  True,  True, False,
        False, False,  True,  True, False,  True,  True,  True, False,  True,
         True,  True,  True,  True, False], device='cuda:0')

In [69]:
        bboxes_preds_per_image = bboxes_preds_per_image[fg_mask]
        cls_preds_ = cls_preds[batch_idx][fg_mask]
        obj_preds_ = obj_preds[batch_idx][fg_mask]
        num_in_boxes_anchor = bboxes_preds_per_image.shape[0]

In [70]:
import torch.nn.functional as F
from yolox.utils import bboxes_iou

In [71]:
gt_bboxes_per_image.shape, bboxes_preds_per_image.shape

(torch.Size([1, 4]), torch.Size([26, 4]))

In [72]:
pair_wise_ious = bboxes_iou(gt_bboxes_per_image, bboxes_preds_per_image, False)

In [73]:
pair_wise_ious.shape

torch.Size([1, 26])

In [74]:
pair_wise_ious

tensor([[-0.0000, -0.0000, 0.0251, -0.0000, -0.0000, -0.0000, -0.0000, -0.0000, -0.0000,
         0.0000, -0.0000, 0.0420, 0.0000, 0.0143, 0.0254, -0.0000, 0.0000, -0.0000,
         -0.0000, -0.0000, -0.0000, 0.0000, 0.0000, -0.0000, -0.0000, -0.0000]],
       device='cuda:0', grad_fn=<DivBackward0>)

In [75]:
        gt_cls_per_image = (
            F.one_hot(gt_classes.to(torch.int64), self.num_classes)
            .float()
        )
        pair_wise_ious_loss = -torch.log(pair_wise_ious + 1e-8)

In [76]:
pair_wise_ious_loss

tensor([[18.4207, 18.4207,  3.6865, 18.4207, 18.4207, 18.4207, 18.4207, 18.4207,
         18.4207, 18.4207, 18.4207,  3.1709, 18.4207,  4.2449,  3.6745, 18.4207,
         18.4207, 18.4207, 18.4207, 18.4207, 18.4207, 18.4207, 18.4207, 18.4207,
         18.4207, 18.4207]], device='cuda:0', grad_fn=<NegBackward0>)

In [77]:
        with torch.amp.autocast('cuda', enabled=False):
            cls_preds_ = (
                cls_preds_.float().sigmoid_() * obj_preds_.float().sigmoid_()
            ).sqrt()
            pair_wise_cls_loss = F.binary_cross_entropy(
                cls_preds_.unsqueeze(0).repeat(num_gt, 1, 1),
                gt_cls_per_image.unsqueeze(1).repeat(1, num_in_boxes_anchor, 1),
                reduction="none"
            ).sum(-1)
        del cls_preds_

In [78]:
pair_wise_cls_loss

tensor([[63.3320, 49.9659, 70.4122, 67.1589, 63.7911, 65.2854, 70.3423, 63.0545,
         71.7550, 60.3692, 77.3189, 76.5813, 68.5446, 68.9083, 80.3012, 75.7691,
         79.5033, 64.7250, 82.7315, 61.1823, 81.0184, 94.4069, 81.3210, 72.5189,
         72.0124, 69.2302]], device='cuda:0', grad_fn=<SumBackward1>)

In [79]:
        cost = (
            pair_wise_cls_loss
            + 3.0 * pair_wise_ious_loss
            + float(1e6) * (~geometry_relation)
        )

In [80]:
cost

tensor([[118.5940, 105.2279,  81.4715, 122.4210, 119.0531, 120.5474, 125.6044,
         118.3166, 127.0170, 115.6312, 132.5810,  86.0939, 123.8066,  81.6431,
          91.3247, 131.0312, 134.7654, 119.9870, 137.9935, 116.4444, 136.2805,
         149.6690, 136.5831, 127.7810, 127.2744, 124.4922]], device='cuda:0',
       grad_fn=<AddBackward0>)

In [49]:
        (
            num_fg,
            gt_matched_classes,
            pred_ious_this_matching,
            matched_gt_inds,
        ) = self.simota_matching(cost, pair_wise_ious, gt_classes, num_gt, fg_mask)

In [81]:
        matching_matrix = torch.zeros_like(cost, dtype=torch.uint8)

        n_candidate_k = min(10, pair_wise_ious.size(1))
        topk_ious, _ = torch.topk(pair_wise_ious, n_candidate_k, dim=1)
        dynamic_ks = torch.clamp((20 * topk_ious.sum(1)).int(), min=1)

In [82]:
dynamic_ks = torch.clamp(dynamic_ks * 0 + n_candidate_k, min=1)

In [83]:
dynamic_ks

tensor([10], device='cuda:0', dtype=torch.int32)

In [84]:
        for gt_idx in range(num_gt):
            _, pos_idx = torch.topk(
                cost[gt_idx], k=dynamic_ks[gt_idx], largest=False
            )
            matching_matrix[gt_idx][pos_idx] = 1

In [85]:
matching_matrix

tensor([[1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0,
         0, 0]], device='cuda:0', dtype=torch.uint8)

In [86]:
        anchor_matching_gt = matching_matrix.sum(0)
        # deal with the case that one anchor matches multiple ground-truths
        if anchor_matching_gt.max() > 1:
            multiple_match_mask = anchor_matching_gt > 1
            _, cost_argmin = torch.min(cost[:, multiple_match_mask], dim=0)
            matching_matrix[:, multiple_match_mask] *= 0
            matching_matrix[cost_argmin, multiple_match_mask] = 1
        fg_mask_inboxes = anchor_matching_gt > 0
        num_fg = fg_mask_inboxes.sum().item()

In [87]:
anchor_matching_gt

tensor([1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0,
        0, 0], device='cuda:0')

In [88]:
num_fg

10

In [75]:
x, y = x_shifts[batch_idx] * expanded_strides[0][0], y_shifts[batch_idx] * expanded_strides[0][1]

In [79]:
x.shape

torch.Size([35])

In [80]:
fg_mask.shape

torch.Size([35])

In [82]:
fg_mask

tensor([False, False, False, False, False, False, False, False, False, False,
        False, False, False, False, False,  True, False, False, False, False,
        False, False, False, False, False, False, False, False, False, False,
        False, False, False, False, False], device='cuda:0')

In [88]:
matching_matrix.shape

torch.Size([1, 17])

In [93]:
x[fg_mask.nonzero()][:, 0][matching_matrix[0]]

/tmp/ipykernel_37307/2888822417.py:1: UserWarning: indexing with dtype torch.uint8 is now deprecated, please use a dtype torch.bool instead. (Triggered internally at /pytorch/aten/src/ATen/native/IndexingUtils.h:29.)
  x[fg_mask.nonzero()][:, 0][matching_matrix[0]]


tensor([51., 50., 53., 54., 66.], device='cuda:0')

In [95]:
x[fg_mask.nonzero()][:, 0]

tensor([118.,  72., 124.,  51.,  85.,  50., 107., 120.,  53.,  54.,  76.,  53.,
         44.,  48.,  66.,  74.,  61.], device='cuda:0')

In [94]:
y[fg_mask.nonzero()][:, 0][matching_matrix[0]]

/tmp/ipykernel_37307/3736721846.py:1: UserWarning: indexing with dtype torch.uint8 is now deprecated, please use a dtype torch.bool instead. (Triggered internally at /pytorch/aten/src/ATen/native/IndexingUtils.h:29.)
  y[fg_mask.nonzero()][:, 0][matching_matrix[0]]


tensor([58., 45., 33., 61., 55.], device='cuda:0')

In [73]:
x_shifts[batch_idx] * expanded_strides[0][0]

tensor([ 22.,  24., 118.,  72., 166.,  65., 187.,  28., 124.,  51.,  85.,  50.,
        107., 120.,  53.,  54., 161.,   6.,  76.,  53.,  44.,  48.,  20., 191.,
          3.,  66.,  74., 165., 163.,  35.,  34., 169., 172.,  29.,  61.],
       device='cuda:0')

In [74]:
y_shifts[batch_idx] * expanded_strides[0][1]

tensor([ 89.0000,  68.0000,  25.0000,  59.0000,  87.0000, 115.0000,  53.0000,
         24.0000,  24.0000,  58.0000,  80.0000,  45.0000,  79.0000,  62.0000,
         33.0000,  61.0000,  48.0000,  94.0000,  37.0000,  31.0000, 100.0000,
         41.0000,  53.0000,  46.0000,  27.0000,  55.0000,  96.0000,  27.0000,
         26.0000,  83.0000,  39.0000,  40.0000,  55.0000,  32.0000,  44.0000],
       device='cuda:0')

In [89]:
cls_targets = []
reg_targets = []
l1_targets = []
obj_targets = []
fg_masks = []

num_fg = 0.0
num_gts = 0.0

for batch_idx in range(outputs.shape[0]):
    num_gt = int(nlabel[batch_idx])
    num_gts += num_gt
    if num_gt == 0:
        cls_target = outputs.new_zeros((0, self.num_classes))
        reg_target = outputs.new_zeros((0, 4))
        l1_target = outputs.new_zeros((0, 4))
        obj_target = outputs.new_zeros((total_num_anchors, 1))
        fg_mask = outputs.new_zeros(total_num_anchors).bool()
    else:
        gt_bboxes_per_image = labels[batch_idx, :num_gt, 1:5]
        gt_classes = labels[batch_idx, :num_gt, 0]
        bboxes_preds_per_image = bbox_preds[batch_idx]
        (
            gt_matched_classes,
            fg_mask,
            pred_ious_this_matching,
            matched_gt_inds,
            num_fg_img,
        ) = self.get_assignments(  # noqa
            batch_idx,
            num_gt,
            gt_bboxes_per_image,
            gt_classes,
            bboxes_preds_per_image,
            expanded_strides,
            x_shifts,
            y_shifts,
            cls_preds,
            obj_preds,
        )

        num_fg += num_fg_img
        print(num_fg_img)
        cls_target = F.one_hot(
            gt_matched_classes.to(torch.int64), self.num_classes
        ) * pred_ious_this_matching.unsqueeze(-1)
        obj_target = fg_mask.unsqueeze(-1)
        reg_target = gt_bboxes_per_image[matched_gt_inds]
        if self.use_l1:
            l1_target = self.get_l1_target(
                outputs.new_zeros((num_fg_img, 4)),
                gt_bboxes_per_image[matched_gt_inds],
                expanded_strides[0][fg_mask],
                x_shifts=x_shifts[0][fg_mask],
                y_shifts=y_shifts[0][fg_mask],
            )

    cls_targets.append(cls_target)
    reg_targets.append(reg_target)
    # obj_targets.append(obj_target.to(dtype))
    fg_masks.append(fg_mask)
    if self.use_l1:
        l1_targets.append(l1_target)

10
10
10
10
10
10
10
10


In [90]:
num_fg

80.0

In [91]:
        cls_targets = torch.cat(cls_targets, 0)
        reg_targets = torch.cat(reg_targets, 0)
        fg_masks = torch.cat(fg_masks, 0)
        if self.use_l1:
            l1_targets = torch.cat(l1_targets, 0)

In [92]:
self.iou_loss(bbox_preds.view(-1, 4)[fg_masks], reg_targets)

tensor([1.0000, 1.0000, 0.9994, 1.0000, 1.0000, 1.0000, 0.9982, 0.9998, 0.9994,
        1.0000, 1.0000, 0.9979, 0.9972, 1.0000, 0.9963, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 0.9987, 0.9977, 0.9988, 0.9995, 0.9971, 0.9986, 0.9998,
        0.9995, 0.9999, 0.9997, 0.9986, 1.0000, 0.9964, 0.9996, 1.0000, 0.9972,
        1.0000, 0.9995, 0.9989, 0.9992, 1.0000, 0.9991, 0.9918, 1.0000, 1.0000,
        1.0000, 0.9998, 1.0000, 1.0000, 1.0000, 0.9990, 0.9955, 0.9914, 0.9967,
        0.9936, 0.9981, 0.9978, 0.9957, 0.9993, 0.9997, 0.9980, 1.0000, 0.9977,
        0.9996, 1.0000, 0.9973, 0.9985, 1.0000, 0.9995, 1.0000, 0.9920, 1.0000,
        0.9915, 0.9907, 0.9939, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
       device='cuda:0', grad_fn=<RsubBackward1>)

In [93]:
(
            self.iou_loss(bbox_preds.view(-1, 4)[fg_masks], reg_targets)
        ).sum() / num_fg

tensor(0.9985, device='cuda:0', grad_fn=<DivBackward0>)

In [40]:
with torch.no_grad():
    model_outputs = model(data)

In [41]:
model_outputs

{'total_loss': tensor(104.3970, device='cuda:0'),
 'iou_loss': tensor(4.9290, device='cuda:0'),
 'l1_loss': 0.0,
 'conf_loss': tensor(25.2903, device='cuda:0'),
 'cls_loss': tensor(74.1776, device='cuda:0'),
 'num_fg': 1.0}